# 🧪 10 · Live-Cell Calibration Practice (Stretch Module)
### Putting this physics/instrumentation sequence to work on one realistic imaging session

**Part B · Seeing like a microscope** · Module 10 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/10_live_cell_calibration_practice.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | Modules 03–09 (the physics/instrumentation sequence) |
| **Estimated time** | 60–75 minutes (optional stretch module) |
| **Instrumentation topics covered** | Photobleaching kinetics; the bleaching/phototoxicity budget alongside the photon budget; uneven illumination (vignetting); flat-field (shading) correction; integrating the instrumentation concepts from Modules 03-09 into one realistic live-cell imaging calibration workflow |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every bleaching curve, illumination field and corrected image here is SIMULATED; no real time-lapse movie or flat-field calibration file is read from disk. This module is an optional stretch module that integrates Modules 03–09 rather than introducing substantial new physics.

## 🎯 Learning objectives

1. Simulate a photobleaching time course and explain the roles of bleach rate and immobile
   fraction.
2. Combine a bleaching tolerance with Module 09's photon-budget framework to design a feasible
   imaging schedule — a maximum tolerable total illumination dose, split across a chosen number of
   time points.
3. Simulate uneven illumination across a field of view and explain its physical causes.
4. Apply flat-field correction to recover a quantitatively accurate image from an unevenly
   illuminated raw image, and verify the correction numerically.
5. Walk through a realistic live-cell imaging calibration checklist that integrates decisions from
   across Modules 03-09 — numerical aperture, aberration tolerance, contrast or fluorescence
   technique, optical sectioning, sensor choice, and the combined photon/bleaching budget.

## 🗺️ Contents

1. Photobleaching kinetics: why fluorescence fades even without obvious damage
2. The bleaching budget meets the photon budget
3. Uneven illumination: why the corners of a field of view are never quite as bright as the centre
4. Flat-field correction: recovering a quantitatively accurate image
5. Putting it together: a one-experiment case study and checklist

## 🧫 The biological question

A live-cell time-lapse of a migrating cell needs to run for several hours, imaging every couple of
minutes, across a field of view wide enough to capture the whole migration path. By the final frame,
the fluorescent reporter looks noticeably dimmer than at the start. Is that real biology — the
reporter being degraded or diluted — genuine photobleaching from repeated excitation, or simply the
fact that the cell wandered from the bright centre of the field of view toward a dimmer corner?
Telling these apart, and designing the experiment so the answer is unambiguous in the first place,
requires putting together nearly everything Modules 03-09 have covered, in the right order, before
data collection even begins — which is exactly what this stretch module practices end to end.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
SEED = 10
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import numpy as np

def photobleaching_curve(frame_times: np.ndarray, initial_intensity: float = 1.0,
                          bleach_rate: float = 0.05, immobile_fraction: float = 0.0) -> np.ndarray:
    """A SIMULATED photobleaching time course: fluorescence intensity decays exponentially with
    cumulative illumination, I(t) = I0 [(1 - f) e^{-k t} + f], where ``bleach_rate`` k sets how
    fast the bleachable fraction decays and ``immobile_fraction`` f is a non-bleaching (or very
    slowly bleaching) population left over at long times."""
    f = immobile_fraction
    return initial_intensity * ((1 - f) * np.exp(-bleach_rate * frame_times) + f)


def photon_budget_tradeoff(total_photon_budget: float, n_frames: np.ndarray) -> dict:
    """The fundamental live-cell imaging trade-off: a fixed total photon budget (set by how much
    bleaching/phototoxicity the sample tolerates) must be divided across ``n_frames`` time
    points. More frames (a longer or finer-grained time-lapse) means fewer photons per frame —
    and therefore lower signal-to-noise, since shot-noise SNR scales as sqrt(photons per frame).
    Returns ``photons_per_frame`` and the resulting ``snr_per_frame`` for each entry of
    ``n_frames``."""
    photons_per_frame = total_photon_budget / n_frames
    snr_per_frame = np.sqrt(photons_per_frame)
    return dict(photons_per_frame=photons_per_frame, snr_per_frame=snr_per_frame)


def simulate_uneven_illumination(size: int = 160, center_bias: float = 0.35) -> np.ndarray:
    """A SIMULATED multiplicative illumination field (vignetting / imperfect Kohler alignment):
    1.0 at the field centre, falling smoothly to ``1 - center_bias`` at the corners — the 'flat
    field' a real flat-fielding calibration image would record."""
    yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing="ij")
    r2 = yy ** 2 + xx ** 2
    return 1.0 - center_bias * r2 / r2.max()


def flat_field_correct(image: np.ndarray, flat_field: np.ndarray, dark_level: float = 0.0) -> np.ndarray:
    """Correct uneven illumination: subtract a dark/offset level from both the image and the
    flat-field calibration image, then divide by the (mean-normalised) flat field — the standard
    shading-correction formula ``(raw - dark) / (flat - dark) * mean(flat - dark)``."""
    flat = flat_field - dark_level
    return (image - dark_level) / (flat + 1e-9) * flat.mean()


def sensor_noise_model(incident_photons: np.ndarray, qe: float = 0.7, read_noise_e: float = 2.0,
                        dark_current_e: float = 0.05, gain: float = 1.0,
                        rng: np.random.Generator | None = None) -> np.ndarray:
    """A simplified camera detection chain: Poisson-sample the photoelectrons actually detected
    (``incident_photons * qe``, the quantum efficiency), add Poisson dark-current electrons, add
    Gaussian read noise (``read_noise_e``, electrons RMS, independent of signal), and apply an
    output ``gain`` (digital counts per electron). Use a low ``read_noise_e`` (~1-2, an EMCCD or
    sCMOS) vs. a higher one (~8-15, an older CCD) to compare sensor types."""
    rng = rng or np.random.default_rng(0)
    photoelectrons = rng.poisson(np.clip(incident_photons, 0, None) * qe).astype(float)
    dark = rng.poisson(dark_current_e, size=incident_photons.shape).astype(float)
    signal_e = photoelectrons + dark
    read_noise = rng.normal(0.0, read_noise_e, size=incident_photons.shape)
    return (signal_e + read_noise) * gain

---
## 1 · Photobleaching kinetics: why fluorescence fades even without obvious damage

Every time a fluorophore absorbs a photon, there is some small probability it undergoes an
irreversible photochemical reaction that permanently destroys its ability to fluoresce again, rather
than simply relaxing and re-emitting as Module 07 described. Across many such absorption events, this
produces a smooth, roughly exponential decline in total fluorescence — **photobleaching** — that
happens even in a perfectly healthy, undamaged specimen, purely as a cumulative consequence of how
much excitation light it has absorbed so far. Some fluorophore populations also have a small
**immobile fraction** that bleaches far more slowly (or not at all) within the experiment's time
scale, producing a plateau rather than complete decay to zero.

In [ ]:
# ── 1.1 photobleaching curves for a few bleach rates and immobile fractions ──
frame_times = np.linspace(0, 100, 300)  # cumulative illumination 'dose' units

fig, ax = plt.subplots(figsize=(7, 3.4))
for bleach_rate, immobile_fraction, label in [
    (0.01, 0.0, 'slow bleaching, no immobile fraction'),
    (0.05, 0.0, 'fast bleaching, no immobile fraction'),
    (0.05, 0.25, 'fast bleaching, 25% immobile fraction'),
]:
    curve = photobleaching_curve(frame_times, initial_intensity=1.0, bleach_rate=bleach_rate,
                                  immobile_fraction=immobile_fraction)
    ax.plot(frame_times, curve, label=label)
ax.set_xlabel('cumulative illumination dose'); ax.set_ylabel('relative fluorescence intensity')
ax.set_title('Photobleaching: a cumulative, not a time-based, effect')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The two "no immobile fraction" curves decay all the way toward zero,
at a rate set entirely by `bleach_rate`; the curve with a 25% immobile fraction instead plateaus at
0.25 rather than continuing to zero. Critically, the x-axis is *cumulative illumination dose*, not
wall-clock time directly — doubling the excitation intensity bleaches a sample in half the time,
because what matters is the total number of excitation events the fluorophore has already absorbed,
not how long the clock has been running.

✏️ **Your turn.** At `bleach_rate = 0.05`, at what cumulative dose does the no-immobile-fraction
curve fall to half its initial value? Does this match the standard exponential-decay half-life
formula $t_{1/2} = \ln(2)/k$?

---
## 2 · The bleaching budget meets the photon budget

Module 09 showed that a fixed total photon budget, divided across more time points, lowers
per-frame SNR. This section adds the other half of the real constraint: the total illumination dose a
specimen can tolerate is itself limited by an acceptable amount of photobleaching (and, in practice,
phototoxicity, which this module treats as scaling similarly with cumulative dose). Setting a
tolerance — for instance, "the reporter should not fall below 70% of its starting intensity by the
final frame" — pins down a maximum total dose, which Module 09's framework then divides among however
many time points the experiment needs.

In [ ]:
# ── 2.1 derive a maximum total dose from a bleaching tolerance, then apply the photon-budget trade-off ──
bleach_rate = 0.02
target_remaining_fraction = 0.70
max_total_dose = -np.log(target_remaining_fraction) / bleach_rate
print(f'Bleach rate: {bleach_rate} per dose-unit')
print(f'Tolerating no more than {1 - target_remaining_fraction:.0%} total bleaching by the final frame')
print(f'=> maximum total illumination dose for the whole movie: {max_total_dose:.1f} dose-units')

photon_yield_per_dose_unit = 400.0  # a calibration constant: detected photons per dose-unit, at the sensor
total_photon_budget = photon_yield_per_dose_unit * max_total_dose

n_frames_range = np.array([10, 25, 50, 100, 200])
budget_result = photon_budget_tradeoff(total_photon_budget, n_frames_range)
for n, snr in zip(n_frames_range, budget_result['snr_per_frame']):
    print(f'{n:>4d} frames across the movie  ->  relative SNR per frame: {snr:7.1f}')

In [ ]:
# ── 2.2 for one chosen schedule, show the TRUE fluorescence intensity declining across the movie ──
n_frames_chosen = 50
dose_per_frame = max_total_dose / n_frames_chosen
frame_index = np.arange(n_frames_chosen)
cumulative_dose = frame_index * dose_per_frame
true_intensity_over_time = photobleaching_curve(cumulative_dose, initial_intensity=1.0, bleach_rate=bleach_rate)

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(frame_index, true_intensity_over_time, color='teal', label='true fluorescence intensity (bleaching only)')
ax.axhline(target_remaining_fraction, color='gray', ls='--', label=f'design tolerance ({target_remaining_fraction:.0%})')
ax.set_xlabel('frame number'); ax.set_ylabel('relative intensity')
ax.set_title(f'Designed schedule: {n_frames_chosen} frames, ending right at the bleaching tolerance')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The chosen schedule (`n_frames_chosen = 50`) was deliberately picked so
that the *true* fluorescence intensity — due to bleaching alone, with no noise or illumination
artifacts yet involved — ends almost exactly at the 70% tolerance line by the final frame, by
construction. This means that even in a perfectly executed experiment, a real and expected 30% decline
in apparent brightness over the course of the movie is simply the cost of imaging at all, before any
noise (Module 09) or illumination artifact (Sections 3-4 below) is layered on top.

✏️ **Your turn.** Looking back at `budget_result`, what relative SNR per frame does
`n_frames_chosen = 50` achieve? If the experiment instead needs at least `SNR = 15` per frame, is 50
frames an acceptable choice, and if not, how many frames would need to be used instead?

---
## 3 · Uneven illumination: why the corners of a field of view are never quite as bright as the centre

Even a well-aligned Kohler illumination system (Module 03) rarely delivers perfectly uniform
light across the entire field of view; lens vignetting, slight misalignment, and the condenser's own
optics typically leave the field centre somewhat brighter than its edges and corners. This
**illumination field** (sometimes called the "flat field," after the flat, featureless calibration
image used to measure it) multiplies the true specimen signal at every pixel, independent of whatever
the specimen itself is doing.

In [ ]:
# ── 3.1 a simulated illumination field, and its effect on an otherwise uniform specimen ──
size = 160
flat_field = simulate_uneven_illumination(size=size, center_bias=0.4)
uniform_specimen = np.ones((size, size))  # a specimen that is, in truth, uniformly bright everywhere
apparent = uniform_specimen * flat_field

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
im0 = axes[0].imshow(flat_field, cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Illumination field (the "flat field")')
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(apparent, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('A truly UNIFORM specimen,\nas it actually appears')
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The specimen in this simulation is, by construction, uniformly bright
everywhere — yet the right-hand panel shows clear, smooth dimming toward the corners, purely because
of the illumination field. If a migrating cell's true fluorescence intensity were constant and it
simply moved from the field centre to a corner during a time-lapse, this is exactly the kind of
artificial dimming that could easily be mistaken for additional photobleaching or real biology,
unless corrected for.

✏️ **Your turn.** Increase `center_bias` to `0.6` (a more strongly vignetted system) and re-plot.
At what point does this artifact become severe enough that it would be obviously visible even without
any quantitative measurement?

---
## 4 · Flat-field correction: recovering a quantitatively accurate image

Because the illumination field multiplies the true signal at every pixel, dividing a raw image
by a (mean-normalized) measurement of that same field exactly undoes the effect — provided the
illumination field itself has been measured, typically from a separate, genuinely featureless
calibration specimen. `flat_field_correct` implements this standard shading-correction formula.

In [ ]:
# ── 4.1 flat-field correction recovers the true, uniform specimen ──
corrected = flat_field_correct(apparent, flat_field)
relative_error_before = np.abs(apparent - uniform_specimen).mean()
relative_error_after = np.abs(corrected - uniform_specimen).mean()
print(f'Mean absolute error vs. true uniform specimen, BEFORE correction: {relative_error_before:.4f}')
print(f'Mean absolute error vs. true uniform specimen, AFTER correction:  {relative_error_after:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(uniform_specimen, cmap='gray', vmin=0, vmax=1.2); axes[0].set_title('Ground truth\n(uniform)')
axes[1].imshow(apparent, cmap='gray', vmin=0, vmax=1.2); axes[1].set_title('Raw (uneven illumination)')
axes[2].imshow(corrected, cmap='gray', vmin=0, vmax=1.2); axes[2].set_title('Flat-field corrected')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The corrected image's mean absolute error against ground truth is many
times smaller than the raw image's — the correction recovers the specimen's true uniformity almost
exactly (limited only by the small numerical offset `flat_field_correct` adds to avoid division by
zero). This is the standard calibration step every quantitative fluorescence-intensity measurement
depends on, and it has to be measured and applied *before* trusting any intensity value's absolute
level, not just its relative pattern.

✏️ **Your turn.** Apply `flat_field_correct` using the *wrong* flat field — for instance
`simulate_uneven_illumination(size=size, center_bias=0.1)`, a much weaker vignetting profile than the
one that actually generated `apparent` — and recompute the mean absolute error. Does an incorrect flat
field make the correction better, worse, or simply different?

---
## 5 · Putting it together: a one-experiment case study and checklist

This section runs the opening biological question directly: a migrating cell's true
fluorescence intensity is declining only due to photobleaching, but it also happens to be moving from
the field centre toward a dimmer corner. Can flat-field correction separate the two effects?

In [ ]:
# ── 5.1 case study: does flat-field correction recover the TRUE bleaching curve from a confounded movie? ──
n_frames_case = 40
t = np.linspace(0, 1, n_frames_case)
cell_y = size * (0.5 + 0.35 * t)   # the cell migrates from the centre toward a corner
cell_x = size * (0.5 + 0.35 * t)

bleach_rate_case = 0.02
dose_per_frame_case = 3.0
true_peak_series = photobleaching_curve(np.arange(n_frames_case) * dose_per_frame_case,
                                         initial_intensity=200.0, bleach_rate=bleach_rate_case)

yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
apparent_peak_raw, apparent_peak_corrected = [], []
for i in range(n_frames_case):
    blob = true_peak_series[i] * np.exp(-(((yy - cell_y[i]) ** 2 + (xx - cell_x[i]) ** 2) / (2 * 6.0 ** 2)))
    illuminated = blob * flat_field
    noisy = sensor_noise_model(illuminated, qe=0.7, read_noise_e=2.0, dark_current_e=0.05, rng=rng)
    corrected_frame = flat_field_correct(noisy, flat_field)
    apparent_peak_raw.append(noisy.max())
    apparent_peak_corrected.append(corrected_frame.max())

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(true_peak_series, color='black', ls='--', lw=2, label='true intensity (bleaching only, ground truth)')
ax.plot(apparent_peak_raw, color='crimson', label='raw measured peak intensity (confounded)')
ax.plot(apparent_peak_corrected, color='teal', label='flat-field CORRECTED peak intensity')
ax.set_xlabel('frame number'); ax.set_ylabel('peak intensity (a.u.)')
ax.set_title('A migrating, bleaching cell: can correction recover the true bleaching curve?')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The raw measured intensity (red) drops noticeably faster than the true
bleaching curve (black dashed), because it is confounded by the cell migrating into the field's
dimmer region on top of real bleaching. The flat-field corrected curve (teal) tracks the true
bleaching curve far more closely, recovering the genuine photobleaching dynamics from underneath the
illumination artifact — directly answering this module's opening question: with the right
calibration step applied, the dimming by the final frame can be confidently attributed to real
photobleaching, not an artifact of where the cell happened to wander.

✏️ **Your turn.** Set `dose_per_frame_case = 0.0` (no bleaching at all) and re-run. Does the raw
(uncorrected) curve still show a decline? What does this confirm about which part of the original
confound was due to illumination and which was due to bleaching?

### 📘 Lecture notes — a live-cell imaging calibration checklist

**Definition.** A reliable quantitative live-cell imaging experiment requires deciding, in roughly
this order, on: (1) objective NA, working distance, and correction class (Modules 03 and 05) for the
required resolution and specimen geometry; (2) contrast or fluorescence technique and filter set
(Modules 06-07) for the specimen's labeling; (3) optical-sectioning method (Module 08) for the
specimen's thickness and the question's time scale; (4) sensor type and bit depth (Module 09) for the
expected light level; (5) a joint photon/bleaching budget (Sections 1-2 of this module) that sets the
feasible number of time points at an acceptable SNR and bleaching tolerance; and (6) a flat-field
calibration measurement (Sections 3-4), taken under the same illumination conditions as the actual
experiment, applied to every frame before any quantitative measurement is trusted.

**Worked example.** Skipping step 6 alone — forgetting flat-field calibration — can make a perfectly
well-designed, bleaching-budget-respecting experiment (steps 1-5 all done correctly) still produce
systematically wrong quantitative conclusions whenever the specimen of interest moves within the
field of view, exactly as Section 5 demonstrated directly.

**One memorable point.** *Every module in Modules 03-10 answers a different piece of "why does this
image look the way it does" — and a real experiment needs all of their answers at once, not just the
one that happens to be most visually obvious.*

## 📌 Key takeaways

- Photobleaching is a cumulative-dose effect, not a simple function of elapsed time; it follows an
  exponential decay, optionally with an immobile, slowly-bleaching fraction.
- A bleaching tolerance and Module 09's photon-budget framework together determine a feasible
  imaging schedule: how many time points at what per-frame SNR a given specimen can support.
- Illumination is rarely perfectly uniform across a field of view; a multiplicative illumination
  field dims the apparent signal toward the corners regardless of true specimen brightness.
- Flat-field correction divides out a measured illumination field, recovering quantitatively
  accurate intensities and letting real biological or photophysical dynamics be distinguished from
  a positional artifact.
- A real live-cell imaging experiment is designed by working through nearly every decision point
  introduced across Modules 03-10, in sequence, before data collection ever begins.

## ✅ Checkpoint

**1.** A researcher doubles their excitation laser power to finish an experiment in half the total
imaging time, keeping the same number of frames. Based on Section 1, does the sample bleach by the
same total amount by the end of the experiment?

<details><summary>Answer</summary>
Yes, assuming the total number of frames and per-frame exposure time are otherwise unchanged — total
bleaching depends on cumulative illumination dose, and doubling power while halving total duration
(for the same frame count) leaves the per-frame dose, and hence the cumulative dose by the final
frame, essentially the same.
</details>

**2.** A quantitative measurement shows a cell's fluorescence declining by 40% over a time-lapse, but
the bleaching-budget calculation (Section 2) predicted only 30% bleaching for the chosen schedule.
What is the first thing to check, based on Sections 3-5?

<details><summary>Answer</summary>
Whether the cell moved within the field of view during the time-lapse, and whether flat-field
correction was applied. Section 5 showed directly that uncorrected illumination unevenness can add
substantial additional apparent decline on top of genuine photobleaching, which would explain a gap
between the predicted and observed decline.
</details>

**3.** Why does the checklist in this section's lecture note place the photon/bleaching budget
(step 5) after the optics and sensor choices (steps 1-4) rather than first?

<details><summary>Answer</summary>
Because the achievable photon budget and the sensor's effective sensitivity depend on the earlier
choices — NA and contrast/fluorescence technique affect how much signal reaches the detector at all,
and sensor choice affects how efficiently that signal is converted and how much noise is added. The
budget calculation needs those earlier decisions as inputs; reversing the order would mean budgeting
against instrumentation that has not actually been chosen yet.
</details>

## 🧠 Questions & discussion

- This module treated phototoxicity as scaling the same way as photobleaching (with cumulative
  illumination dose). Are there situations where this assumption might break down — for instance,
  where cell health is affected by instantaneous light intensity rather than total dose?
- A multi-position time-lapse experiment revisits several different fields of view in sequence,
  cycling back to each one periodically. How does the bleaching-budget framework from Section 2
  need to be adapted for this common real-world acquisition pattern?
- Flat-field correction in this module used a single, fixed illumination field for the whole
  experiment. What would need to be true for the illumination field itself to change over the course
  of a long time-lapse, and how would you detect that it had?

## 🚀 Bonus — going further

- Combine this module's bleaching-budget calculation with Module 08's confocal vs. multiphoton
  comparison: since multiphoton confines excitation (and therefore bleaching) to the focal volume,
  how might its effective "bleaching budget" differ for a thick 3-D specimen compared to confocal
  imaging of the same specimen?
- Measure a flat field directly from a simulated calibration image (a uniform specimen plus sensor
  noise, rather than the clean analytic field used in this module) and see how sensor noise
  (Module 09) affects the quality of the resulting correction.
- Extend Section 5's case study to a specimen with its own real intensity change over time (for
  instance, a genuine biological signaling event) superimposed on bleaching and illumination
  artifacts, and see whether flat-field correction plus knowledge of the bleaching rate can recover
  the genuine signal change.

## 📝 Lab note

```text
Module: 10 — Live-Cell Calibration Practice (Stretch Module)
Date:
The bleaching tolerance and resulting imaging schedule I would choose for MY own experiment:
One calibration step from this module I would now perform before trusting a quantitative result:
```

## 📚 References

1. Diaspro, A. (Ed.) (2010). *Optical Fluorescence Microscopy: From the Spectral to the Nano
   Dimension*. Springer.
2. Waters, J.C. (2009). Accuracy and precision in quantitative fluorescence microscopy. *Journal of
   Cell Biology*, 185(7), 1135-1148.
3. Model, M.A. (2014). Intensity calibration and flat-field correction for fluorescence microscopes.
   *Current Protocols in Cytometry*, 68(1), 10.14.1-10.14.10.
4. Murphy, D.B., Davidson, M.W. (2012). *Fundamentals of Light Microscopy and Electronic Imaging*
   (2nd ed.). Wiley-Blackwell. — a fuller treatment of photobleaching and image calibration practice.

## ➡️ Next

This stretch module completes the physics and instrumentation sequence that began back in Module 02.
Module 11 ("Filtering and Histograms") now returns to Part B's original computational thread,
carrying forward a grounded understanding of exactly what physical and instrumentation choices
shaped every pixel in every image this course analyzes from here on.